# Chapter 4: What Agents Remember

This notebook mirrors the chapter. Run the cells in order, from the `Chapter04`
folder. Nothing here needs an API key. The SDK runner, the sessions, the tools,
the MCP server, and the guardrail are all real. Only the model's replies are
scripted, and embeddings come from a small local embedder.

Pinned version: openai-agents 0.24.0.

In [1]:
import sys
from datetime import UTC, datetime, timedelta
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
print("openai-agents", version("openai-agents"))

openai-agents 0.24.0


## Working memory: What fits in the context window

### Where the past actually lives: Sessions

Two turns through one file-backed session. The scripted model reports how many
earlier items the session placed in front of it on the second turn.

In [2]:
from examples import billing_session
from examples.scripted import scripted_billing_model

print(await billing_session.main(scripted_billing_model(), ":memory:"))

The session handed me 2 earlier items. You had told me: "I was double charged in March." Let us pick up from there.


## Semantic memory: Giving the agent facts it can look up

### The pipeline as a tool the agent reaches for

Index two passages, then search. Every result carries its source.

In [3]:
from chapter4.embedding import local_embed
from chapter4.memory import MemoryContext, MemoryScope
from chapter4.stores import EpisodicStore, KnowledgeStore

scope = MemoryScope(org_id="st-marys", user_id="patient-alvarez", agent_id="med-rec")
memory = MemoryContext(scope, local_embed, KnowledgeStore(), EpisodicStore())

for source, text in [
    ("policy/refunds-eu", "Refunds for European orders take five days."),
    ("policy/shipping", "We ship on weekdays."),
]:
    memory.knowledge.add(source, text, await local_embed(text))

query = await local_embed("refunds for European orders")
for hit in await memory.knowledge.nearest(query, limit=2):
    print(f"{hit.score:.2f}  {hit.source:20} {hit.text}")

0.76  policy/refunds-eu    Refunds for European orders take five days.
0.00  policy/shipping      We ship on weekdays.


## Episodic memory: Remembering this person, over time

### Store the meaning, not the tape

Redaction runs before anything is embedded or stored.

In [4]:
from chapter4.episodes import redact

print(redact("Patient MRN 4471982 reported a severe reaction to amoxicillin."))
print(redact("Dose raised to 40 mg after 180 days."))

Patient [identifier removed] reported a severe reaction to amoxicillin.
Dose raised to 40 mg after 180 days.


### Scope is the line between a feature and a breach

One episode is stored for one patient at one hospital. The same search from
another organization, or for another patient, returns nothing.

In [5]:
from chapter4.stores import Episode

now = datetime.now(UTC)
stopped = Episode(
    summary="Simvastatin stopped by Dr. Reed.",
    importance=8,
    occurred_at=now - timedelta(days=10),
)
await memory.episodes.append(scope, await local_embed(stopped.summary), stopped)

vector = await local_embed("simvastatin")
other_org = scope.model_copy(update={"org_id": "other-hospital"})
other_user = scope.model_copy(update={"user_id": "patient-brown"})
for label, who in [("same scope", scope), ("other org", other_org), ("other user", other_user)]:
    print(f"{label:11} {len(await memory.episodes.search(who, vector, 5))} result(s)")

same scope  1 result(s)
other org   0 result(s)
other user  0 result(s)


### Recency, importance, and the art of the gentle discount

Three memories: an old trivial one, a recent one, and an old allergy stored with
`decays=False`. The allergy keeps its full weight.

In [6]:
from chapter4.episodes import rank_by_relevance_and_recency, score
from chapter4.stores import EpisodeHit


def hit(text, days_old, similarity, importance=5, decays=True):
    when = now - timedelta(days=days_old)
    episode = Episode(summary=text, importance=importance, occurred_at=when, decays=decays)
    return EpisodeHit(episode, similarity)


hits = [
    hit("Asked about opening hours.", 700, 0.80),
    hit("Asked to move the appointment.", 3, 0.75),
    hit("Severe reaction to amoxicillin.", 700, 0.70, importance=10, decays=False),
]
for h in hits:
    print(f"{score(h, now):.3f}  {h.episode.summary}")
print()
print([e.summary for e in rank_by_relevance_and_recency(hits)])

0.003  Asked about opening hours.
0.550  Asked to move the appointment.
0.700  Severe reaction to amoxicillin.

['Severe reaction to amoxicillin.', 'Asked to move the appointment.', 'Asked about opening hours.']


## Procedural memory: Remembering how

### Skills: Procedures you can hold in your hand

The catalog costs one line per skill. The body is loaded only when it is needed.

In [7]:
from chapter4.skills import describe_skills, load_skill

print(describe_skills())
print()
print(load_skill("medication-reconciliation"))

{'medication-reconciliation': "Reconcile a patient's medications against pharmacy fills before a visit."}

Steps:
1. Fetch active medications from the patient record via the FHIR tool.
2. Fetch the last 180 days of pharmacy fills.
3. Align by drug and dose. Flag duplicates, gaps, and discontinued-but-filled.
4. Recall prior reconciliations for this patient before flagging anything twice.
5. Check the surviving medications against the interaction knowledge base.
6. Draft a reconciliation note for the clinician to review. Do not finalize it.


## The discipline of forgetting

### Deleting when the law says you must

An erasure request takes one record out of retrieval. The record under a legal
hold stays, and both rows still exist until the hard purge.

In [8]:
store = EpisodicStore()
morning = Episode(summary="Prefers morning appointments.", importance=3, occurred_at=now)
statin = Episode(summary="Statin decision recorded.", importance=8, occurred_at=now)
await store.append(scope, await local_embed(morning.summary), morning)
await store.append(scope, await local_embed(statin.summary), statin, legal_hold=True)

print("tombstoned:", store.erase("st-marys", "patient-alvarez"))
left = await store.search(scope, await local_embed("appointments statin"), 5)
print("still retrievable:", [h.episode.summary for h in left])
print("rows still stored:", len(store.rows))

tombstoned: 1
still retrievable: ['Statin decision recorded.']
rows still stored: 2


## Putting it together: A medication reconciliation agent

### One turn, traced

The agent starts the FHIR server as a separate process, reads the record and the
pharmacy fills over MCP, recalls the last reconciliation, checks interactions,
and returns a draft. The scripted model plays the clinician's assistant.

In [9]:
from examples import reconcile
from examples.scripted import scripted_clinician_model

seen = {}
model = scripted_clinician_model("Draft for clinician review.", seen)
draft = await reconcile.main(model, local_embed, "sqlite+aiosqlite:///:memory:")

print(draft.status.upper(), "for", draft.patient_ref)
for finding in draft.findings:
    print(" -", finding)
print()
print("recalled episode:", seen["past"][:120])

DRAFT for patient-alvarez
 - Apixaban with ibuprofen raises bleeding risk.
 - Single statin is intended. Simvastatin was stopped at the last visit.

recalled episode: [Episode(summary='Duplicate statin resolved. Simvastatin stopped by Dr. Reed. Atorvastatin continues.', importance=8, oc


### Where the rules bite, and where it breaks

A note that presents itself as final trips the output guardrail.

In [10]:
from agents import OutputGuardrailTripwireTriggered

model = scripted_clinician_model("Final note, signed.", {})
try:
    await reconcile.main(model, local_embed, "sqlite+aiosqlite:///:memory:")
except OutputGuardrailTripwireTriggered:
    print("Guardrail tripped: the agent may only return a draft.")

Guardrail tripped: the agent may only return a draft.
